# MLflow Repo Pipeline (v2, resumable on Drive)
Run cells top to bottom. Every step checkpoints to Google Drive, so a dead Colab session costs nothing: just rerun Setup (Cells 1-2) and the step you were on.

**Pipeline:** SEART export → license filter → contamination date filter → MLflow pre-filter (manifests) → file detector (AST) → 10-sample validation.

## 1. Setup: mount Drive first (always run)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/mlflow_research'
os.makedirs(PROJECT, exist_ok=True)
print("Project folder:", PROJECT)
print("Contents:", os.listdir(PROJECT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/mlflow_research
Contents: ['mlflow_files.csv', 'mlflow_files.gsheet', 'results.csv.gz', 'candidates_licensed.csv', 'candidates.csv', 'mlflow_repos.csv', 'prefilter_processed.txt']


## 2. Setup: GitHub token (always run)

In [3]:
from getpass import getpass
GITHUB_TOKEN = getpass("Paste the token Corey sent, then press Enter: ")
print("Token captured, length:", len(GITHUB_TOKEN))

Paste the token Corey sent, then press Enter: ··········
Token captured, length: 93


## 3. One-time: load SEART export into Drive
Upload `results.csv.gz` ONCE. It's copied to Drive so you never re-upload after a session drop. Skip this cell if `results.csv.gz` is already in the project folder.

In [ ]:
import shutil, os
from google.colab import files

if os.path.exists(f"{PROJECT}/results.csv.gz"):
    print("results.csv.gz already in Drive, skipping upload.")
else:
    uploaded = files.upload()
    shutil.move("results.csv.gz", f"{PROJECT}/results.csv.gz")
    print("Saved to Drive.")

results.csv.gz already in Drive, skipping upload.


## 4. License filter

In [ ]:
import pandas as pd

df = pd.read_csv(f"{PROJECT}/results.csv.gz")
print("Rows:", len(df))

keep = [
    "MIT License",
    "Apache License 2.0",
    'BSD 2-Clause "Simplified" License',
    'BSD 3-Clause "New" or "Revised" License',
]
filtered = df[df["license"].isin(keep)]

print("Before license filter:", len(df))
print("After license filter:", len(filtered))
print(filtered["license"].value_counts())

filtered.to_csv(f"{PROJECT}/candidates_licensed.csv", index=False)

Rows: 15739
Before license filter: 15739
After license filter: 9154
license
MIT License           6628
Apache License 2.0    2526
Name: count, dtype: int64


## 5. Contamination date filter (NEW)
Models: Qwen 3 8B (released 2025-04-29) and Llama 3.1 8B (released 2024-07-23). Cutoff = the later date. Keep only repos created after it.

Note: confirm with Corey whether to use the repo *created* date or *last commit* date. This cell uses created date (conservative). If the column names below don't match your SEART export, the first print shows what's available.

In [ ]:
import pandas as pd

CUTOFF = pd.Timestamp("2025-04-29", tz="UTC")   # Qwen 3 release (later than Llama 3.1)

df = pd.read_csv(f"{PROJECT}/candidates_licensed.csv")
print("Columns:", list(df.columns))

DATE_COL = "createdAt"   # change if your export names it differently
df[DATE_COL] = pd.to_datetime(df[DATE_COL], utc=True, errors="coerce")

before = len(df)
out = df[df[DATE_COL] > CUTOFF]
print(f"Before date filter: {before}")
print(f"After date filter:  {len(out)}")
print(f"Removed:            {before - len(out)}")

out.to_csv(f"{PROJECT}/candidates.csv", index=False)

Columns: ['id', 'name', 'isFork', 'commits', 'branches', 'releases', 'forks', 'mainLanguage', 'defaultBranch', 'license', 'homepage', 'watchers', 'stargazers', 'contributors', 'size', 'createdAt', 'pushedAt', 'updatedAt', 'totalIssues', 'openIssues', 'totalPullRequests', 'openPullRequests', 'blankLines', 'codeLines', 'commentLines', 'metrics', 'lastCommit', 'lastCommitSHA', 'hasWiki', 'isArchived', 'isDisabled', 'isLocked', 'languages', 'labels', 'topics']
Before date filter: 9154
After date filter:  9154
Removed:            0


## 6. MLflow pre-filter (resumable, checkpoints to Drive)
Scans each repo's root manifest files for "mlflow". Safe to interrupt: rerun this cell and it continues where it stopped. Progress files live in Drive.

In [ ]:
import base64, time, os, requests, pandas as pd

MANIFESTS = {
    "requirements.txt","requirements-dev.txt","pyproject.toml","setup.py",
    "setup.cfg","environment.yml","environment.yaml","Pipfile","conda.yaml",
}

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})

def gh_get(url):
    while True:
        r = session.get(url)
        if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
            wait = max(int(r.headers.get("X-RateLimit-Reset", time.time()+60)) - int(time.time()) + 1, 1)
            print(f"  rate limit, sleeping {wait}s"); time.sleep(wait); continue
        return r

def root_files(full):
    r = gh_get(f"https://api.github.com/repos/{full}/contents")
    return {i["name"] for i in r.json() if i.get("type") == "file"} if r.status_code == 200 else None

def mentions_mlflow(full, fname):
    r = gh_get(f"https://api.github.com/repos/{full}/contents/{fname}")
    if r.status_code != 200: return False
    d = r.json()
    if d.get("encoding") != "base64": return False
    try: return "mlflow" in base64.b64decode(d["content"]).decode("utf-8","ignore").lower()
    except Exception: return False

PROC = f"{PROJECT}/prefilter_processed.txt"
OUT  = f"{PROJECT}/mlflow_repos.csv"

all_repos = pd.read_csv(f"{PROJECT}/candidates.csv")["name"].dropna().tolist()

processed = set()
if os.path.exists(PROC):
    processed = {l.strip() for l in open(PROC) if l.strip()}
if not os.path.exists(OUT):
    open(OUT, "w").write("repo,evidence_file\n")

repos = [r for r in all_repos if r not in processed]
kept = sum(1 for _ in open(OUT)) - 1
print(f"Total {len(all_repos)}, done {len(processed)}, remaining {len(repos)}, kept so far {kept}\n")

out_f = open(OUT, "a"); proc_f = open(PROC, "a")
for i, full in enumerate(repos, 1):
    root = root_files(full)
    if root:
        for m in (MANIFESTS & root):
            if mentions_mlflow(full, m):
                out_f.write(f"{full},{m}\n"); out_f.flush(); kept += 1
                print(f"KEEP {full} ({m})  total kept: {kept}"); break
    proc_f.write(full + "\n"); proc_f.flush()
    if i % 200 == 0: print(f"[{i}/{len(repos)}] kept total: {kept}")
    time.sleep(0.03)
out_f.close(); proc_f.close()
print(f"\nFinished. Total MLflow repos: {kept}")

Total 9154, done 6780, remaining 2374, kept so far 41

  rate limit, sleeping 1156s
[200/2374] kept total: 41
KEEP kkruglik/mlflow-mcp (pyproject.toml)  total kept: 42
[400/2374] kept total: 42
[600/2374] kept total: 42
KEEP murtiunlimited/face-emotion-recognition (requirements.txt)  total kept: 43
KEEP opendatahub-io/agent-eval-harness (pyproject.toml)  total kept: 44
[800/2374] kept total: 44
KEEP NVIDIA-NeMo/nemo-platform (pyproject.toml)  total kept: 45
[1000/2374] kept total: 45
KEEP ai-infra-curriculum/ai-infra-engineer-learning (requirements.txt)  total kept: 46
KEEP eduardocornelsen/full-funnel-ai-analytics (requirements.txt)  total kept: 47
KEEP Tejas-TA/predikit (pyproject.toml)  total kept: 48
[1200/2374] kept total: 48
[1400/2374] kept total: 48
KEEP SuperagenticAI/superqode (pyproject.toml)  total kept: 49
KEEP WhitzardAgent/qitos (setup.py)  total kept: 50
[1600/2374] kept total: 50
[1800/2374] kept total: 50
KEEP genalyu/WM-R1 (requirements.txt)  total kept: 51
[2000/237

## 7. File detector (resumable, checkpoints to Drive)
For each MLflow repo, finds Python files that import/call mlflow via GitHub code search + AST parsing. Also safe to interrupt and rerun.

In [4]:
import ast, base64, time, requests, csv, os
import pandas as pd

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"})

def gh_get(url):
    while True:
        r = session.get(url)
        if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
            wait = max(int(r.headers.get("X-RateLimit-Reset", time.time()+60)) - int(time.time()) + 1, 1)
            print(f"  rate limit, sleeping {wait}s"); time.sleep(wait); continue
        return r

def code_search(full):
    paths, page = [], 1
    while True:
        r = gh_get(f"https://api.github.com/search/code?q=mlflow+repo:{full}+language:python&per_page=100&page={page}")
        if r.status_code != 200:
            print(f"  search failed {full}: {r.status_code}"); break
        items = r.json().get("items", [])
        paths += [it["path"] for it in items]
        if len(items) < 100: break
        page += 1; time.sleep(7)
    return paths

def fetch(full, path):
    r = gh_get(f"https://api.github.com/repos/{full}/contents/{path}")
    if r.status_code != 200: return None
    d = r.json()
    if d.get("encoding") != "base64": return None
    try: return base64.b64decode(d["content"]).decode("utf-8","ignore")
    except Exception: return None

def analyze(src):
    has_import, n_calls = False, 0
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return ("import mlflow" in src or "from mlflow" in src), src.count("mlflow.")
    for node in ast.walk(tree):
        if isinstance(node, ast.Import) and any(n.name=="mlflow" or n.name.startswith("mlflow.") for n in node.names):
            has_import = True
        elif isinstance(node, ast.ImportFrom) and node.module and (node.module=="mlflow" or node.module.startswith("mlflow.")):
            has_import = True
        elif isinstance(node, ast.Call):
            f = node.func
            while isinstance(f, ast.Attribute):
                if isinstance(f.value, ast.Name) and f.value.id=="mlflow":
                    n_calls += 1; break
                f = f.value
    return has_import, n_calls

PROC = f"{PROJECT}/detector_processed.txt"
OUT  = f"{PROJECT}/mlflow_files.csv"

repos = pd.read_csv(f"{PROJECT}/mlflow_repos.csv")["repo"].dropna().tolist()

processed = set()
if os.path.exists(PROC):
    processed = {l.strip() for l in open(PROC) if l.strip()}
if not os.path.exists(OUT):
    with open(OUT, "w", newline="") as f:
        csv.writer(f).writerow(["repo","file_path","has_import","n_calls"])

todo = [r for r in repos if r not in processed]
print(f"Total {len(repos)}, done {len(processed)}, remaining {len(todo)}\n")

out_f = open(OUT, "a", newline=""); w = csv.writer(out_f)
proc_f = open(PROC, "a")
total_files = 0
for full in todo:
    print(f"\n{full}")
    paths = code_search(full)
    print(f"  {len(paths)} candidate file(s)")
    for p in paths:
        src = fetch(full, p)
        if not src: continue
        imp, calls = analyze(src)
        if imp or calls:
            w.writerow([full, p, imp, calls]); out_f.flush(); total_files += 1
            print(f"  MLflow file: {p}  (import={imp}, calls={calls})")
    proc_f.write(full + "\n"); proc_f.flush()
    time.sleep(7)
out_f.close(); proc_f.close()
print(f"\nDone. {total_files} new MLflow files recorded. Saved to Drive.")

Total 52, done 0, remaining 52


tier4/autoware-ml
  14 candidate file(s)
  MLflow file: autoware_ml/utils/mlflow_store.py  (import=True, calls=1)
  MLflow file: autoware_ml/utils/mlflow_helpers.py  (import=True, calls=0)
  MLflow file: autoware_ml/scripts/deploy.py  (import=True, calls=0)
  MLflow file: autoware_ml/tests/utils/test_mlflow_store.py  (import=True, calls=0)
  MLflow file: autoware_ml/tests/utils/test_mlflow_utils.py  (import=True, calls=0)

Cazzy-Aporbo/PearlMind-ML-Journey
  3 candidate file(s)
  MLflow file: docker_development.py  (import=False, calls=2)
  MLflow file: src/pearlmind/experiment/tracker.py  (import=True, calls=6)

sunnynguyen-ai/fraud-detection-system
  0 candidate file(s)

ScopeX-ASU/MAPS
  0 candidate file(s)

hpccube/OneScience
  0 candidate file(s)

Abraham-Einstein/MAFS
  0 candidate file(s)

radixark/miles
  5 candidate file(s)
  MLflow file: miles/utils/tracking_utils/mlflow_utils.py  (import=True, calls=10)

NVIDIA-AI-Blueprints/ai-model-distilla

<unknown>:289: SyntaxWarning: invalid escape sequence '\.'



msaad00/agent-bom
  37 candidate file(s)
  MLflow file: src/agent_bom/cloud/mlflow_provider.py  (import=True, calls=0)

microsoft/physical-ai-toolchain
  41 candidate file(s)
  MLflow file: evaluation/metrics/bootstrap_mlflow.py  (import=True, calls=2)
  MLflow file: training/utils/context.py  (import=True, calls=2)
  MLflow file: training/utils/aml_mirror.py  (import=True, calls=15)
  MLflow file: training/rl/scripts/launch.py  (import=True, calls=1)
  MLflow file: training/rl/scripts/launch_rsl_rl.py  (import=True, calls=1)
  MLflow file: evaluation/metrics/upload_artifacts.py  (import=True, calls=10)
  MLflow file: training/il/scripts/lerobot/bootstrap.py  (import=True, calls=3)
  MLflow file: training/il/scripts/lerobot/checkpoints.py  (import=True, calls=2)
  MLflow file: training/il/scripts/lerobot/train.py  (import=True, calls=8)
  MLflow file: evaluation/sil/scripts/run_evaluation.py  (import=True, calls=10)
  MLflow file: training/rl/scripts/rsl_rl/train.py  (import=True, cal

## 8. 10-sample validation (Dr. Abdellatif's request)
Draws a reproducible random sample of 10 repos from the final list. Manually check each on GitHub (search the repo for `import mlflow`) and compare against the pipeline's rows shown below. Record results in your validation spreadsheet.

In [6]:
import pandas as pd, random

files_df = pd.read_csv(f"{PROJECT}/mlflow_files.csv")
final_repos = sorted(files_df["repo"].unique())
print(f"Final repos with MLflow files: {len(final_repos)}\n")

random.seed(42)
sample = random.sample(final_repos, 10)

for r in sample:
    rows = files_df[files_df["repo"] == r]
    print(f"=== {r}  →  https://github.com/{r}")
    for _, row in rows.iterrows():
        print(f"    {row['file_path']}  (import={row['has_import']}, calls={row['n_calls']})")
    print()

Final repos with MLflow files: 33

=== Qredence/fleet-rlm  →  https://github.com/Qredence/fleet-rlm
    scripts/mlflow_cli.py  (import=True, calls=2)
    src/fleet_rlm/quality/mlflow_evaluation.py  (import=True, calls=0)
    src/fleet_rlm/integrations/observability/mlflow_traces.py  (import=True, calls=6)
    src/fleet_rlm/integrations/observability/mlflow_runtime.py  (import=True, calls=9)
    src/fleet_rlm/integrations/observability/mlflow_context.py  (import=False, calls=3)
    src/fleet_rlm/quality/scorers.py  (import=True, calls=0)
    src/fleet_rlm/quality/eval/evaluate.py  (import=True, calls=10)
    src/fleet_rlm/api/bootstrap_observability.py  (import=True, calls=0)
    scripts/evaluate_rlm_capabilities.py  (import=True, calls=6)
    src/fleet_rlm/api/routers/optimization/background.py  (import=True, calls=0)
    src/fleet_rlm/integrations/observability/auto_assessment.py  (import=True, calls=0)

=== CognicellAI/Cognition  →  https://github.com/CognicellAI/Cognition
    server

## 9. Summary counts (for progress email)

In [7]:
import pandas as pd, os

raw = pd.read_csv(f"{PROJECT}/results.csv.gz")
lic = pd.read_csv(f"{PROJECT}/candidates_licensed.csv")
dated = pd.read_csv(f"{PROJECT}/candidates.csv")
kept = pd.read_csv(f"{PROJECT}/mlflow_repos.csv")
print(f"SEART export:              {len(raw)}")
print(f"After license filter:      {len(lic)}")
print(f"After date filter:         {len(dated)}")
print(f"After MLflow pre-filter:   {len(kept)}")
if os.path.exists(f"{PROJECT}/mlflow_files.csv"):
    files_df = pd.read_csv(f"{PROJECT}/mlflow_files.csv")
    print(f"Repos with MLflow files:   {files_df['repo'].nunique()}")
    print(f"Total MLflow files found:  {len(files_df)}")

SEART export:              15739
After license filter:      9154
After date filter:         9154
After MLflow pre-filter:   52
Repos with MLflow files:   33
Total MLflow files found:  112


In [ ]:
import pandas as pd
df = pd.read_csv(f"{PROJECT}/candidates.csv")
d = pd.to_datetime(df["createdAt"], utc=True, errors="coerce")
print("earliest createdAt:", d.min())
print("latest createdAt:", d.max())
print("null dates:", d.isna().sum())

earliest createdAt: 2025-08-31 01:06:18+00:00
latest createdAt: 2026-06-16 12:27:19+00:00
null dates: 0


In [8]:
import pandas as pd
f = pd.read_csv(f"{PROJECT}/mlflow_files.csv")
print("duplicate rows:", f.duplicated().sum())
f2 = f.drop_duplicates()
print("unique files:", len(f2), "| unique repos:", f2["repo"].nunique())

duplicate rows: 7
unique files: 105 | unique repos: 33


In [9]:
import pandas as pd
f = pd.read_csv(f"{PROJECT}/mlflow_files.csv")
f.drop_duplicates().to_csv(f"{PROJECT}/mlflow_files.csv", index=False)
f = pd.read_csv(f"{PROJECT}/mlflow_files.csv")
print("rows now:", len(f), "| duplicates:", f.duplicated().sum())

rows now: 105 | duplicates: 0
